# Кейс № 8 — подбор гиперпараметров (пункт 5)
**Проектный практикум · команда 13**

**Задача:** оптимизировать регрессионную модель для прогноза числа арендованных велосипедов за час.

Это **первый самостоятельный шаблон**: пока модели других участников не добавлены в `main`, используем `RandomForestRegressor` как пример. Когда команда определит 1–2 лучшие модели, заменим `model` и `param_distributions` на их варианты.

**Правила эксперимента:**
- только хронологический `train → validation → test`, без `shuffle`;
- `TimeSeriesSplit` внутри **train** для подбора гиперпараметров;
- `validation` — для сравнения исходной и настроенной моделей;
- **test не используем** в этом ноутбуке;
- категориальные признаки преобразуем внутри `Pipeline`, чтобы избежать утечки информации.

**Важно:** параметры сплита и политика обработки нерабочих часов ниже временные — их требуется согласовать с командой до итоговых экспериментов.

## 1. Импорты и настройки

`random_state=42` фиксирует случайность **обучения модели и поиска**. Хронологическое разбиение само по себе детерминированное и не требует `random_state`.

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
TRAIN_FRACTION = 0.70  # временно, до согласования с командой
VALID_FRACTION = 0.15  # оставшиеся 15% — тест
KEEP_NON_FUNCTIONING = True  # временно: нерабочие часы остаются в данных

# Если CSV уже скачан, можно указать путь вручную, например:
# LOCAL_CSV = Path("data/raw/SeoulBikeData.csv")
LOCAL_CSV = None

pd.set_option("display.max_columns", 30)

## 2. Загрузка фиксированной версии данных

В `eda.ipynb` Александра датасет загружается через `kagglehub` из `joebeachcapital/seoul-bike-sharing`. Здесь указан **version 1**. Если локального файла нет, код попробует скачать эту версию.

Сохраняем SHA-256 файла — это цифровой отпечаток, который поможет проверить, что все участники используют одинаковые данные.

In [ ]:
def project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return cwd

root = project_root()
if LOCAL_CSV is not None:
    csv_path = Path(LOCAL_CSV).expanduser().resolve()
else:
    local_candidates = [
        root / "data" / "raw" / "SeoulBikeData.csv",
        root / "files" / "SeoulBikeData.csv",
    ]
    csv_path = next((f for f in local_candidates if f.is_file()), None)
    if csv_path is None:
        import kagglehub
        downloaded_path = kagglehub.dataset_download(
            "joebeachcapital/seoul-bike-sharing/versions/1"
        )
        matches = list(Path(downloaded_path).rglob("SeoulBikeData.csv"))
        if not matches:
            raise FileNotFoundError("SeoulBikeData.csv не найден в каталоге Kaggle")
        csv_path = matches[0]

if not csv_path.is_file():
    raise FileNotFoundError(f"Датасет не найден: {csv_path}")

file_hash = hashlib.sha256(csv_path.read_bytes()).hexdigest()
print("CSV:", csv_path)
print("SHA-256:", file_hash)

try:
    df = pd.read_csv(csv_path, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(csv_path, encoding="cp1252")
print("Размер:", df.shape)
assert len(df) == 8760, "Ожидали 8760 почасовых наблюдений — проверьте версию CSV"
df.head(3)

## 3. Подготовка временной оси и признаков

Это **минимальная временная реализация**, чтобы можно было проверить пункт 5 до готовности общего кода команды. Позже заменим на подготовку данных Андрея, сохранив одинаковый формат `X_train`, `y_train`, `X_val`, `y_val`.

Мы **не используем `Rented Bike Count` для создания признаков**. `Date` переводим в день недели/месяц, а время объединяем в `timestamp` только для сортировки.

In [ ]:
required_columns = {
    "Date", "Hour", "Rented Bike Count", "Seasons", "Holiday", "Functioning Day"
}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"В CSV отсутствуют колонки: {missing_columns}")

frame = df.copy()
frame["Date"] = pd.to_datetime(frame["Date"], format="%d/%m/%Y")
frame["timestamp"] = frame["Date"] + pd.to_timedelta(frame["Hour"], unit="h")
frame = frame.sort_values("timestamp").reset_index(drop=True)

if frame["timestamp"].duplicated().any():
    raise ValueError("Есть повторяющиеся дата + час")

frame["day_of_week"] = frame["Date"].dt.dayofweek
frame["month"] = frame["Date"].dt.month
frame["is_weekend"] = (frame["day_of_week"] >= 5).astype(int)
frame["hour_sin"] = np.sin(2 * np.pi * frame["Hour"] / 24)
frame["hour_cos"] = np.cos(2 * np.pi * frame["Hour"] / 24)

print("Период:", frame["timestamp"].min(), "—", frame["timestamp"].max())
print("Нерабочих часов:", (frame["Functioning Day"] == "No").sum())
frame[["timestamp", "Hour", "day_of_week", "is_weekend", "hour_sin", "hour_cos"]].head()

## 4. Один общий хронологический сплит

Делим **по индексам отсортированного по времени датасета**. У модели нет доступа к информации из будущих периодов на этапе обучения.

Политика `KEEP_NON_FUNCTIONING` применяется **после разбиения**, чтобы при изменении политики временные границы оставались одинаковыми. Пока по умолчанию сохраняем все часы. Это техническое допущение, **не принятое командой решение**.

In [ ]:
n = len(frame)
train_end = int(n * TRAIN_FRACTION)
val_end = train_end + int(n * VALID_FRACTION)
assert 0 < train_end < val_end < n

train_data = frame.iloc[:train_end].copy()
val_data = frame.iloc[train_end:val_end].copy()
test_data = frame.iloc[val_end:].copy()  # НЕ используем для тюнинга

if not KEEP_NON_FUNCTIONING:
    train_data = train_data.loc[train_data["Functioning Day"] == "Yes"].copy()
    val_data = val_data.loc[val_data["Functioning Day"] == "Yes"].copy()
    # test_data оставляем нетронутой до итоговой проверки команды

print("TRAIN:", len(train_data), train_data["timestamp"].min(), train_data["timestamp"].max())
print("VALID:", len(val_data), val_data["timestamp"].min(), val_data["timestamp"].max())
print("TEST: ", len(test_data), "наблюдений (заморожена; метрики не считаем)")
assert train_data["timestamp"].max() < val_data["timestamp"].min()
assert val_data["timestamp"].max() < test_data["timestamp"].min()

## 5. Отбор признаков и конвейер обработки

Все преобразования категорий (`OneHotEncoder`) обучаются **только на train** внутри `Pipeline`. Деревьям не требуется масштабирование числовых признаков.

Для Ridge и других линейных моделей будет нужен **другой** препроцессор со `StandardScaler`, согласованный с ответственным за эти модели.

In [ ]:
target_column = "Rented Bike Count"
cat_cols = ["Seasons", "Holiday", "Functioning Day"]
num_cols = [
    "Hour", "Temperature(°C)", "Humidity(%)", "Wind speed (m/s)",
    "Visibility (10m)", "Dew point temperature(°C)",
    "Solar Radiation (MJ/m2)", "Rainfall(mm)", "Snowfall (cm)",
    "day_of_week", "month", "is_weekend", "hour_sin", "hour_cos",
]
feature_cols = num_cols + cat_cols
missing_features = set(feature_cols) - set(frame.columns)
if missing_features:
    raise ValueError(f"Не найдены признаки: {missing_features}")

X_train = train_data[feature_cols]
y_train = train_data[target_column]
X_val = val_data[feature_cols]
y_val = val_data[target_column]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop",
)

print("Признаки:", len(feature_cols))
print("X_train:", X_train.shape, "X_val:", X_val.shape)

## 6. Функция расчёта метрик

`RMSE` — корень из **среднего квадрата** ошибки: сильнее штрафует большие промахи. `MAE` — средняя абсолютная ошибка. `R²` отражает качество относительно константного прогноза. Чем ниже RMSE/MAE — тем лучше.

In [ ]:
def regression_metrics(y_true, y_pred) -> dict:
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }

def show_metrics(label: str, metrics: dict) -> None:
    print(f"{label}: RMSE={metrics['RMSE']:.2f}, "
          f"MAE={metrics['MAE']:.2f}, R²={metrics['R2']:.3f}")

## 7. Исходная модель — точка отсчёта

Это временный пример. В дальнейшем используем именно **лучшую модель от участника, отвечающего за ансамбли**, сохранив её настройки и совместимый `Pipeline`.

Сейчас для демонстрации берём `RandomForestRegressor` из `scikit-learn`.

In [ ]:
base_model = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=None,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )),
])

base_model.fit(X_train, y_train)
base_val_pred = np.maximum(base_model.predict(X_val), 0)
base_metrics = regression_metrics(y_val, base_val_pred)
show_metrics("Исходная модель (validation)", base_metrics)

## 8. TimeSeriesSplit: правильная кросс-валидация

Это несколько проверок модели, в которых каждый раз обучение происходит **раньше по времени**, чем проверка. Делается **только на train**.

`gap=24` исключает 24 часа между обучающей и проверочной частями каждого фолда. Это консервативная защита от соседства наблюдений, хотя она сама по себе не решает все проблемы временных данных.

In [ ]:
tscv = TimeSeriesSplit(n_splits=4, gap=24)
for fold, (idx_fit, idx_cv) in enumerate(tscv.split(X_train), start=1):
    print(f"Фолд {fold}: train={len(idx_fit)} строк, "
          f"CV={len(idx_cv)} строк; "
          f"последний train idx={idx_fit[-1]}, первый CV idx={idx_cv[0]}")
    assert idx_fit[-1] < idx_cv[0]

## 9. Подбор гиперпараметров через RandomizedSearchCV

Гиперпараметры — настройки алгоритма **до обучения**. Например, `max_depth` ограничивает глубину деревьев, а `min_samples_leaf` — минимальный размер листа.

`RandomizedSearchCV` берёт несколько комбинаций из заданного набора и оценивает их с помощью `TimeSeriesSplit`. Параметры начинаются с `model__`, потому что модель находится внутри `Pipeline` под именем `model`.

**Сейчас это небольшой учебный поиск** (`n_iter=6`), чтобы его можно было запустить на ноутбуке. Для окончательного эксперимента число итераций можно увеличить после согласования вычислительного бюджета.

In [ ]:
param_distributions = {
    "model__n_estimators": [80, 120, 180],
    "model__max_depth": [8, 14, None],
    "model__min_samples_leaf": [1, 3, 6],
    "model__max_features": [0.7, 1.0],
}

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_distributions,
    n_iter=6,
    scoring="neg_root_mean_squared_error",  # sklearn максимизирует score
    cv=tscv,
    refit=True,   # повторное обучение лучшей конфигурации на всём train
    random_state=RANDOM_STATE,
    n_jobs=2,
    verbose=1,
    return_train_score=True,
    error_score="raise",
)
search.fit(X_train, y_train)

print("Лучшие гиперпараметры:", search.best_params_)
print("Средний RMSE по CV на train:", round(-search.best_score_, 2))

## 10. Просмотр всех испытанных комбинаций

Не берём просто наименьший train-RMSE: так легко выбрать переобученную модель. Ранжируем по **CV-RMSE** из нескольких временных фолдов.

In [ ]:
cv_table = pd.DataFrame(search.cv_results_)
cv_table["CV_RMSE"] = -cv_table["mean_test_score"]
cv_table["Train_RMSE"] = -cv_table["mean_train_score"]
cv_table = cv_table.sort_values("rank_test_score")
display(cv_table[[
    "rank_test_score", "CV_RMSE", "Train_RMSE", "std_test_score", "params"
]].head(10))

## 11. Проверка эффекта тюнинга на validation

**Validation не участвовала в подборе параметров.** На ней сравниваем исходную и настроенную модели. Если стало хуже — это честный результат, а не повод подбирать параметры по test.

In [ ]:
best_model = search.best_estimator_
tuned_val_pred = np.maximum(best_model.predict(X_val), 0)
tuned_metrics = regression_metrics(y_val, tuned_val_pred)
show_metrics("После тюнинга (validation)", tuned_metrics)

comparison = pd.DataFrame([
    {"Модель": "До тюнинга", **base_metrics},
    {"Модель": "После тюнинга", **tuned_metrics},
]).set_index("Модель")
display(comparison.round(3))

rmse_gain = base_metrics["RMSE"] - tuned_metrics["RMSE"]
print(f"Изменение RMSE: {rmse_gain:+.2f} велосипедов (плюс означает улучшение)")

## 12. Проверка переобучения

Сравниваем ошибку лучшей модели на train и validation. Сильный разрыв (train гораздо лучше) — признак потенциального переобучения. Небольшой разрыв сам по себе не доказывает, что всё хорошо: из-за смены сезонов данные могут отличаться.

In [ ]:
best_train_pred = np.maximum(best_model.predict(X_train), 0)
train_metrics = regression_metrics(y_train, best_train_pred)
show_metrics("Train, лучшая модель", train_metrics)
show_metrics("Validation, лучшая модель", tuned_metrics)

## 13. График ошибок по часам (только validation)

Проверяем, в какие часы модель ошибается сильнее. Это важно для практического смысла проекта: ошибки в час пик могут быть особенно существенными.

In [ ]:
validation_errors = pd.DataFrame({
    "hour": val_data["Hour"].to_numpy(),
    "actual": y_val.to_numpy(),
    "predicted": tuned_val_pred,
})
validation_errors["abs_error"] = (
    validation_errors["actual"] - validation_errors["predicted"]
).abs()

hour_mae = validation_errors.groupby("hour")["abs_error"].mean()
plt.figure(figsize=(11, 4))
hour_mae.plot(kind="bar")
plt.title("MAE по часам суток — настроенная модель, validation")
plt.xlabel("Час суток")
plt.ylabel("Средняя абсолютная ошибка, велосипедов")
plt.tight_layout()
plt.show()

## 14. Что передать команде и как заменить учебную модель

**Готовые результаты для пункта 5:**
1. список лучших гиперпараметров `search.best_params_`;
2. средний CV-RMSE на `train` (`-search.best_score_`);
3. таблица RMSE/MAE/R² до и после тюнинга **на validation**;
4. вывод о переобучении и ошибках по часам.

**Дальше вместе с участником пункта 4:**
- заменить `RandomForestRegressor` на выбранную командой модель, например `CatBoostRegressor`, либо взять готовый командный `Pipeline`;
- заменить словарь `param_distributions` на параметры именно этой модели;
- использовать **те же train / validation** и признаки, что у остальных;
- не выполнять пункт 7 (финальный test) раньше согласованного решения команды.

**Нерабочие часы:** если команда примет решение обучаться только на `Functioning Day == Yes`, согласованно изменить `KEEP_NON_FUNCTIONING` и повторить **все** результаты. Не смешивать метрики разных политик.

**Данные и исходный код команды:** https://github.com/Vatman30000/Project_Practicum_team_13